In [1]:
import os
import yaml
import copy
import time
import numpy as np
import pandas as pd
import xarray as xr

In [2]:
import matplotlib.pyplot as plt
%matplotlib inline

### Preparing gridded yearly metrics

In [3]:
dict_loc = {
    'Pituffik': (76.4, -68.575),
    'Fairbanks': (64.75, -147.4),
    'Guam': (13.475, 144.75),
    'Yuma_PG': (33.125, -114.125),
    'Fort_Bragg': (35.05, -79.115),
}
keys = list(dict_loc.keys())

### Pituffik: melting degree days

In [4]:
key = 'Pituffik'
dir_stn = f'/glade/derecho/scratch/ksha/EPRI_data/METRICS/{key}/'
base_dir = '/glade/derecho/scratch/ksha/EPRI_data/CESM2_SMYLE/'

In [49]:
list_MDD_all = []
thres = 273.15

for year in range(1958, 2020):
    
    # get data and variable
    fn_CESM = base_dir + f'/{key}/SMYLE_{key}_{year}.zarr'
    ds_CESM = xr.open_zarr(fn_CESM)[["TREFHT"]]
    ds_CESM = ds_CESM.sel(time=slice(f'{year+1}-01-01T00', f'{year+10}-12-31T00'))

    list_MDD_lead = []
    for lead_year in range(year+1, year+10+1):
        ds_ = ds_CESM.sel(time=slice(f"{lead_year}-03-01", f"{lead_year}-06-30"))
        mdd_mj = (ds_["TREFHT"] - thres).clip(min=0).sum(dim="time")
        ds_MDD = mdd_mj.to_dataset(name="MDD").assign_coords(year=lead_year).expand_dims("year")
        list_MDD_lead.append(ds_MDD)
        
    ds_MDD_lead = xr.merge(list_MDD_lead)
    ds_MDD_lead = ds_MDD_lead.assign_coords(year=np.arange(10))
    list_MDD_all.append(ds_MDD_lead)
    
ds_MDD_all = xr.concat(list_MDD_all, dim='init_time')
ds_MDD_all = ds_MDD_all.assign_coords({'init_time': np.arange(1958+1, 2020+1)})
ds_MDD_all = ds_MDD_all.chunk({'init_time': 62, 'year': 10, 'lat': 21, 'lon': 16})

In [28]:
save_name = dir_stn + 'MDD.zarr'
# ds_MDD_all.to_zarr(save_name, mode='w')
print(save_name)

/glade/derecho/scratch/ksha/EPRI_data/METRICS/Pituffik/MDD.zarr


### Pituffik: Freeze-Thaw days

In [30]:
list_FT_all = []
Tf = 273.15  # K

for year in range(1958, 2020):
    
    # get data and variable
    fn_CESM = base_dir + f'/{key}/SMYLE_{key}_{year}.zarr'
    ds_CESM = xr.open_zarr(fn_CESM)[["TREFHTMX", "TREFHTMN"]]
    ds_CESM = ds_CESM.sel(time=slice(f'{year+1}-01-01T00', f'{year+10}-12-31T00'))

    list_FT_lead = []
    for lead_year in range(year+1, year+10+1):
        ds_ = ds_CESM.sel(time=slice(f"{lead_year}-03-01", f"{lead_year}-06-30"))
        # Freeze–Thaw day mask (1 if crossing, else 0)
        ft_day = ((ds_["TREFHTMN"] < Tf) & (ds_["TREFHTMX"] > Tf)).astype("int8")
        # Count days in the season
        ft_count = ft_day.sum(dim="time")
        ds_FT = ft_count.to_dataset(name="FT_days").assign_coords(year=lead_year).expand_dims("year")
        list_FT_lead.append(ds_FT)
        
    ds_FT_lead = xr.merge(list_FT_lead)
    ds_FT_lead = ds_FT_lead.assign_coords(year=np.arange(10))
    list_FT_all.append(ds_FT_lead)
    
ds_FT_all = xr.concat(list_FT_all, dim='init_time')
ds_FT_all = ds_FT_all.assign_coords({'init_time': np.arange(1958+1, 2020+1)})
ds_FT_all = ds_FT_all.chunk({'init_time': 62, 'year': 10, 'lat': 21, 'lon': 16})

In [31]:
ds_FT_all

<xarray.Dataset>
Dimensions:    (lat: 22, lon: 16, year: 10, init_time: 62)
Coordinates:
  * lat        (lat) float64 66.44 67.38 68.32 69.27 ... 83.4 84.35 85.29 86.23
  * lon        (lon) float64 282.5 283.8 285.0 286.2 ... 297.5 298.8 300.0 301.2
  * year       (year) int64 0 1 2 3 4 5 6 7 8 9
  * init_time  (init_time) int64 1959 1960 1961 1962 ... 2017 2018 2019 2020
Data variables:
    FT_days    (init_time, year, lat, lon) float64 dask.array<chunksize=(62, 10, 21, 16), meta=np.ndarray>

In [32]:
save_name = dir_stn + 'FT.zarr'
ds_FT_all.to_zarr(save_name, mode='w')
print(save_name)

/glade/derecho/scratch/ksha/EPRI_data/METRICS/Pituffik/FT.zarr
